# TRIAGE-EG E2E-G1 - Safe Hypothesis Preservation + Metric-Aware Coverage

Runs exactly `G0_E2E1_COARSE`, `G1_COVERAGE_COARSE`, and `G2_SAFE_M1` on TEAM-EVAL development data. Cross runs first; all prediction files are finalized, hashed, and validated before GT is loaded. No SEALED content, model download, VLM, Agent, Event Graph, M2, M3, or parameter sweep.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from time import monotonic
from zipfile import ZIP_DEFLATED, ZipFile, ZipInfo

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path(os.environ.get("AIC_REPO_DIR", "/kaggle/working/AIC2026_TeamPTK_SGU"))
DATA_INPUT = Path(os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic"))
TEAM_EVAL_INPUT = Path(
    os.environ.get(
        "AIC_TEAM_EVAL_DEV_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-team-eval-dev-v1"
    )
)
STAGE1_INPUT = Path(
    os.environ.get(
        "AIC_STAGE1_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle"
    )
)
STAGE1B_INPUT = Path(
    os.environ.get(
        "AIC_STAGE1B_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports",
    )
)
STAGE1E_INPUT = Path(
    os.environ.get(
        "AIC_STAGE1E_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-stage1e-language-path-freeze",
    )
)
CLIP_INPUT = Path(
    os.environ.get("AIC_CLIP_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32")
)
OPUS_INPUT = Path(
    os.environ.get("AIC_OPUS_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en")
)
HISTORICAL_INPUT = Path(
    os.environ.get(
        "AIC_HISTORICAL_E2E1_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-e2e1-v01-bundle"
    )
)
RUN_DEV_L21_150 = True  # Frozen experiment contract: Cross first, then L21.
OUTPUT_ROOT = Path("/kaggle/working/triage_eg_e2eg1_v01")
RUNTIME_ROOT = Path("/kaggle/working/triage_eg_e2eg1_stage2_runtime")
ZIP_PATH = Path("/kaggle/working/triage_eg_e2eg1_v01_bundle.zip")
TEAM_EVAL_REPACKED_ZIP = Path("/kaggle/working/aic2026_team_eval_dev_v1_repacked.zip")
EXTRACT_ROOT = Path("/kaggle/working/aic2026_team_eval_dev_v1_extracted")
INFERENCE_ROOT = Path("/kaggle/working/triage_eg_e2eg1_inference_only")
MATERIALIZED_ROOTS = {
    "stage1": Path("/kaggle/working/triage_eg_stage1_materialized"),
    "stage1b": Path("/kaggle/working/triage_eg_stage1b_materialized"),
    "stage1e": Path("/kaggle/working/triage_eg_stage1e_materialized"),
    "clip": Path("/kaggle/working/aic2026_openai_clip_materialized"),
    "opus": Path("/kaggle/working/aic2026_opus_materialized"),
}
for target in (
    OUTPUT_ROOT,
    RUNTIME_ROOT,
    EXTRACT_ROOT,
    INFERENCE_ROOT,
    *MATERIALIZED_ROOTS.values(),
):
    if target.exists():
        if target.parent != Path("/kaggle/working"):
            raise RuntimeError(f"Refusing cleanup outside /kaggle/working: {target}")
        shutil.rmtree(target)
for target in (ZIP_PATH, TEAM_EVAL_REPACKED_ZIP):
    target.unlink(missing_ok=True)
print(
    {
        "required_inputs": {
            "raw_dataset": str(DATA_INPUT),
            "team_eval_dev_bundle": str(TEAM_EVAL_INPUT),
            "stage1_exact_index": str(STAGE1_INPUT),
            "stage1b_verified_contract": str(STAGE1B_INPUT),
            "stage1e_language_contract": str(STAGE1E_INPUT),
            "openai_clip_offline_asset": str(CLIP_INPUT),
            "opus_mt_vi_en_offline_asset": str(OPUS_INPUT),
        },
        "optional_input": {"historical_e2e1_bundle": str(HISTORICAL_INPUT)},
        "internet_required": "ONLY_FOR_GIT_CLONE_IF_REPO_NOT_PRESENT",
        "model_download_required": False,
        "output_zip": str(ZIP_PATH),
    }
)

In [ ]:
SEARCH_ROOT = Path("/kaggle/input")
MAX_DEPTH, MAX_DIRECTORIES = 7, 10000
TEAM_EVAL_REQUIRED_MEMBERS = (
    "README.md",
    "benchmark_registry.json",
    "benchmarks/dev_cross_60/annotation_audit.jsonl",
    "benchmarks/dev_cross_60/gt.jsonl",
    "benchmarks/dev_cross_60/manifest.json",
    "benchmarks/dev_cross_60/queries.jsonl",
    "benchmarks/dev_l21_150/annotation_audit.jsonl",
    "benchmarks/dev_l21_150/gt.jsonl",
    "benchmarks/dev_l21_150/manifest.json",
    "benchmarks/dev_l21_150/queries.jsonl",
)


def bounded_dirs(root):
    queue, visited = [(Path(root), 0)], 0
    while queue:
        current, depth = queue.pop(0)
        if not current.is_dir():
            continue
        visited += 1
        if visited > MAX_DIRECTORIES:
            raise RuntimeError("Kaggle input discovery exceeded bound")
        yield current
        if depth < MAX_DEPTH:
            queue.extend(
                (child, depth + 1)
                for child in sorted(current.iterdir())
                if child.is_dir() and not child.is_symlink()
            )


def resolve_file(hint, filename, *, optional=False):
    hint = Path(hint)
    if hint.is_file() and hint.name == filename:
        return hint.resolve()
    roots = [hint] if hint.exists() else [SEARCH_ROOT]
    matches = []
    for root in roots:
        matches.extend(
            directory / filename
            for directory in bounded_dirs(root)
            if (directory / filename).is_file()
        )
        if matches:
            break
    matches = sorted(set(path.resolve() for path in matches))
    if not matches and optional:
        return None
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one {filename}; found {matches}")
    return matches[0]


def is_team_eval_root(root):
    root = Path(root)
    return root.is_dir() and all((root / member).is_file() for member in TEAM_EVAL_REQUIRED_MEMBERS)


def resolve_team_eval_root(hint, *, optional=False):
    hint = Path(hint)
    roots = [hint] if hint.exists() else [SEARCH_ROOT]
    matches = []
    for root in roots:
        matches.extend(
            directory for directory in bounded_dirs(root) if is_team_eval_root(directory)
        )
        if matches:
            break
    matches = sorted(set(path.resolve() for path in matches))
    if not matches and optional:
        return None
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one extracted TEAM-EVAL root; found {matches}")
    return matches[0]


def resolve_dataset(hint):
    hint = Path(hint)
    marker = Path("map-keyframes-aic25-b1/map-keyframes")
    roots = [hint] if hint.exists() else [SEARCH_ROOT]
    matches = []
    for root in roots:
        matches.extend(
            directory
            for directory in bounded_dirs(root)
            if (directory / marker).is_dir() and any(directory.glob("Videos_*/video"))
        )
        if matches:
            break
    matches = sorted(set(path.resolve() for path in matches))
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one raw dataset root; found {matches}")
    return matches[0]


DATASET_ROOT = resolve_dataset(DATA_INPUT)
TEAM_EVAL_MOUNTED_ZIP = resolve_file(TEAM_EVAL_INPUT, "aic2026_team_eval_dev_v1.zip", optional=True)
TEAM_EVAL_MOUNTED_ROOT = resolve_team_eval_root(TEAM_EVAL_INPUT, optional=True)
HISTORICAL_E2E1_ZIP = resolve_file(HISTORICAL_INPUT, "triage_eg_e2e1_v01_bundle.zip", optional=True)
print(
    {
        "raw_dataset": str(DATASET_ROOT),
        "team_eval_zip": str(TEAM_EVAL_MOUNTED_ZIP) if TEAM_EVAL_MOUNTED_ZIP else None,
        "team_eval_extracted_root": str(TEAM_EVAL_MOUNTED_ROOT) if TEAM_EVAL_MOUNTED_ROOT else None,
        "historical_e2e1_zip": str(HISTORICAL_E2E1_ZIP)
        if HISTORICAL_E2E1_ZIP
        else "NOT_MOUNTED_OPTIONAL",
    }
)

In [ ]:
if not (REPO_DIR / ".git").is_dir():
    if REPO_DIR.exists():
        raise RuntimeError(f"Incomplete repository directory: {REPO_DIR}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
        env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
    )
if not (REPO_DIR / "src/triage_eg/e2eg1/pipeline.py").is_file():
    raise RuntimeError(
        "TRIAGEEG ref does not contain E2E-G1; publish reviewed source changes "
        "before Kaggle execution"
    )
sys.path.insert(0, str(REPO_DIR / "src"))
HEAD = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
BRANCH = subprocess.run(
    ["git", "branch", "--show-current"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
GIT_STATUS = subprocess.run(
    ["git", "status", "--short"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
if BRANCH != REPO_REF:
    raise RuntimeError(f"Expected branch {REPO_REF}, resolved {BRANCH}")
print({"branch": BRANCH, "HEAD": HEAD, "git_status": GIT_STATUS or "CLEAN"})

In [ ]:
import yaml

from aic2026_eval.io import sha256_file
from triage_eg.e2eg1 import E2EG1Settings
from triage_eg.retrieval.stage1b.inputs import resolve_stage1_root
from triage_eg.retrieval.stage1d.inputs import resolve_input_root


def repack_team_eval_root(source_root, destination):
    source_root, destination = Path(source_root), Path(destination)
    missing = [
        member for member in TEAM_EVAL_REQUIRED_MEMBERS if not (source_root / member).is_file()
    ]
    if missing:
        raise RuntimeError(f"Extracted TEAM-EVAL bundle incomplete: {missing}")
    destination.unlink(missing_ok=True)
    with ZipFile(destination, "w", compression=ZIP_DEFLATED) as archive:
        for member in TEAM_EVAL_REQUIRED_MEMBERS:
            if "sealed" in member.casefold():
                raise RuntimeError("SEALED_CONTENT_REJECTED")
            info = ZipInfo(member, date_time=(1980, 1, 1, 0, 0, 0))
            info.compress_type = ZIP_DEFLATED
            info.external_attr = 0o644 << 16
            archive.writestr(info, (source_root / member).read_bytes())
    return destination.resolve(strict=True)


if TEAM_EVAL_MOUNTED_ZIP:
    TEAM_EVAL_ZIP, TEAM_EVAL_SOURCE = TEAM_EVAL_MOUNTED_ZIP, "KAGGLE_ZIP"
elif TEAM_EVAL_MOUNTED_ROOT:
    TEAM_EVAL_ZIP = repack_team_eval_root(TEAM_EVAL_MOUNTED_ROOT, TEAM_EVAL_REPACKED_ZIP)
    TEAM_EVAL_SOURCE = "KAGGLE_EXTRACTED_BUNDLE_REPACKED"
else:
    raise RuntimeError("Missing finalized TEAM-EVAL development ZIP or extracted bundle root")


def optional_search_root(hint):
    return None if Path(hint).exists() else SEARCH_ROOT


STAGE1_ROOT = resolve_stage1_root(
    STAGE1_INPUT,
    search_root=optional_search_root(STAGE1_INPUT),
    materialize_root=MATERIALIZED_ROOTS["stage1"],
)
STAGE1B_ROOT, _ = resolve_input_root(
    STAGE1B_INPUT,
    required=(
        "stage1b_summary.json",
        "encoder/selected_encoder_contract.json",
        "encoder/runtime_adapter_manifest.json",
    ),
    materialize_root=MATERIALIZED_ROOTS["stage1b"],
    search_root=optional_search_root(STAGE1B_INPUT),
    archive_keyword="stage1b",
)
STAGE1E_ROOT, _ = resolve_input_root(
    STAGE1E_INPUT,
    required=("stage1e_summary.json", "language_path_contract.json"),
    materialize_root=MATERIALIZED_ROOTS["stage1e"],
    search_root=optional_search_root(STAGE1E_INPUT),
    archive_keyword="stage1e",
)
CLIP_ROOT, _ = resolve_input_root(
    CLIP_INPUT,
    required=("checkpoint/ViT-B-32.pt", "manifests/asset_manifest.json"),
    materialize_root=MATERIALIZED_ROOTS["clip"],
    search_root=optional_search_root(CLIP_INPUT),
    archive_keyword="clip",
)
OPUS_ROOT, _ = resolve_input_root(
    OPUS_INPUT,
    required=("model/config.json", "manifests/asset_manifest.json"),
    materialize_root=MATERIALIZED_ROOTS["opus"],
    search_root=optional_search_root(OPUS_INPUT),
    archive_keyword="opus",
)
SETTINGS = E2EG1Settings()
E2EG1_CONFIG = yaml.safe_load(
    (REPO_DIR / "configs/experiments/triage_e2eg1_v01.yaml").read_text(encoding="utf-8")
)
CONFIG_SETTINGS = {key: E2EG1_CONFIG[key] for key in SETTINGS.as_dict()}
if CONFIG_SETTINGS != SETTINGS.as_dict() or E2EG1_CONFIG.get("parameter_sweep") is not False:
    raise RuntimeError("Frozen E2E-G1 YAML/dataclass contract mismatch")
EXPECTED_HISTORICAL_SHA256 = "606f4cb8910281df55e25f02048d1424d53d30b73190153b3aac5ac323ec141a"
if HISTORICAL_E2E1_ZIP and sha256_file(HISTORICAL_E2E1_ZIP) != EXPECTED_HISTORICAL_SHA256:
    raise RuntimeError("Historical E2E-1 artifact SHA-256 mismatch")
GPU_POLICY = yaml.safe_load(
    (REPO_DIR / "configs/retrieval/gpu_g11_frozen_policy.yaml").read_text(encoding="utf-8")
)
print(
    {
        "team_eval": str(TEAM_EVAL_ZIP),
        "team_eval_sha256": sha256_file(TEAM_EVAL_ZIP),
        "team_eval_source": TEAM_EVAL_SOURCE,
        "stage1": str(STAGE1_ROOT),
        "stage1b": str(STAGE1B_ROOT),
        "stage1e": str(STAGE1E_ROOT),
        "clip": str(CLIP_ROOT),
        "opus": str(OPUS_ROOT),
    }
)
print("FROZEN_G1_CONFIG=", json.dumps(SETTINGS.as_dict(), indent=2))
print("GPU_POLICY=", json.dumps(GPU_POLICY, indent=2))

In [ ]:
from triage_eg.e2eg1 import extract_development_bundle

TEAM_EVAL_ROOT = extract_development_bundle(TEAM_EVAL_ZIP, EXTRACT_ROOT)
with ZipFile(TEAM_EVAL_ZIP) as archive:
    MEMBERS = archive.namelist()
assert not any("sealed" in name.casefold() for name in MEMBERS)
CROSS_ROOT = TEAM_EVAL_ROOT / "benchmarks/dev_cross_60"
L21_ROOT = TEAM_EVAL_ROOT / "benchmarks/dev_l21_150"
print({"cross": str(CROSS_ROOT), "l21": str(L21_ROOT), "SEALED_ACCESS_GATE": "PASS"})

In [ ]:
from triage_eg.e2eg1 import SafeCoveragePipeline
from triage_eg.retrieval.stage2 import config_from_yaml

STARTUP_STARTED = monotonic()
STAGE2_CONFIG = config_from_yaml(
    REPO_DIR / "configs/retrieval/stage2_operational_runtime_gpu.yaml",
    stage1_root=STAGE1_ROOT,
    stage1b_root=STAGE1B_ROOT,
    stage1e_root=STAGE1E_ROOT,
    clip_asset_root=CLIP_ROOT,
    translator_asset_root=OPUS_ROOT,
    output_root=RUNTIME_ROOT,
    stage1d_config=REPO_DIR / "configs/retrieval/stage1d_translation_ablation.yaml",
    build_git_commit=HEAD,
)
PIPELINE = SafeCoveragePipeline.load_once(STAGE2_CONFIG, DATASET_ROOT, settings=SETTINGS)
STARTUP_SECONDS = monotonic() - STARTUP_STARTED
print(
    {
        "startup_seconds": STARTUP_SECONDS,
        "resources": PIPELINE.runtime.runtime_manifest(),
        "M2": False,
        "M3": False,
        "EVENT_GRAPH": False,
        "VLM": False,
        "AGENT": False,
        "NVDEC_DEFAULT": False,
    }
)

In [ ]:
from triage_eg.e2eg1 import materialize_inference_only, run_prediction_variant

CROSS_INFERENCE = materialize_inference_only(CROSS_ROOT, INFERENCE_ROOT / "dev_cross_60")
assert {path.name for path in CROSS_INFERENCE.iterdir()} == {"queries.jsonl"}
print(
    {
        "cross_inference_only": str(CROSS_INFERENCE),
        "members": [path.name for path in CROSS_INFERENCE.iterdir()],
    }
)

In [ ]:
CROSS_G0 = run_prediction_variant(
    PIPELINE, CROSS_INFERENCE, "DEV_CROSS_60", "G0_E2E1_COARSE", OUTPUT_ROOT
)


def verify_historical_g0_kis(run, member):
    if HISTORICAL_E2E1_ZIP is None:
        return "NOT_MOUNTED_OPTIONAL"
    with ZipFile(HISTORICAL_E2E1_ZIP) as archive:
        historical = [
            json.loads(line)
            for line in archive.read(member).decode("utf-8").splitlines()
            if line
        ]
    kis_ids = {row["query_id"] for row in run["queries"] if row["task"] == "KIS"}
    expected = [row for row in historical if row["query_id"] in kis_ids]
    actual = [row for row in run["predictions"] if row["query_id"] in kis_ids]
    if actual != expected:
        raise RuntimeError(f"Historical G0 KIS mismatch for {member}")
    return {"queries": len(kis_ids), "prediction_rows": len(actual), "status": "PASS"}


CROSS_HISTORICAL_G0 = verify_historical_g0_kis(
    CROSS_G0, "predictions/dev_cross_60_p0_coarse_predictions.jsonl"
)
print(
    {
        "variant": CROSS_G0["variant"],
        "predictions": len(CROSS_G0["predictions"]),
        "sha256": CROSS_G0["sha256"],
        "validation": CROSS_G0["validation"],
        "historical_g0_kis": CROSS_HISTORICAL_G0,
    }
)

In [ ]:
CROSS_G1 = run_prediction_variant(
    PIPELINE, CROSS_INFERENCE, "DEV_CROSS_60", "G1_COVERAGE_COARSE", OUTPUT_ROOT
)
print(
    {
        "variant": CROSS_G1["variant"],
        "predictions": len(CROSS_G1["predictions"]),
        "sha256": CROSS_G1["sha256"],
        "validation": CROSS_G1["validation"],
    }
)

In [ ]:
from triage_eg.e2eg1 import combine_prediction_variants

CROSS_G2 = run_prediction_variant(
    PIPELINE, CROSS_INFERENCE, "DEV_CROSS_60", "G2_SAFE_M1", OUTPUT_ROOT
)
CROSS_RUN = combine_prediction_variants(CROSS_G0, CROSS_G1, CROSS_G2)
assert all(value["validation"]["status"] == "PASS" for value in CROSS_RUN["variants"].values())
assert all(
    value["sha256"] == sha256_file(value["prediction_path"])
    for value in CROSS_RUN["variants"].values()
)
print(
    {
        "variant": CROSS_G2["variant"],
        "predictions": len(CROSS_G2["predictions"]),
        "sha256": CROSS_G2["sha256"],
        "PREDICTION_CONTRACT_GATE": "PASS",
        "GT_LEAKAGE_GATE": "PASS",
    }
)

In [ ]:
from triage_eg.e2eg1 import evaluate_finalized, post_inference_diagnostics

# Cross GT is first loaded inside these calls, after all G0/G1/G2 hashes exist.
CROSS_EVAL = evaluate_finalized(CROSS_RUN, CROSS_ROOT, "DEV_CROSS_60", OUTPUT_ROOT)
CROSS_POST = post_inference_diagnostics(
    CROSS_RUN, CROSS_EVAL, CROSS_ROOT, "DEV_CROSS_60", OUTPUT_ROOT
)
EVALUATIONS = {"DEV_CROSS_60": CROSS_EVAL}
PREDICTION_RUNS = {"DEV_CROSS_60": CROSS_RUN}
POST = {"DEV_CROSS_60": CROSS_POST}
print(json.dumps({variant: value["summary"] for variant, value in CROSS_EVAL.items()}, indent=2))
assert (OUTPUT_ROOT / "evaluation/cross_g2_summary.json").is_file()
print("DEV_CROSS_60 persisted before L21")

In [ ]:
if RUN_DEV_L21_150:
    L21_INFERENCE = materialize_inference_only(L21_ROOT, INFERENCE_ROOT / "dev_l21_150")
    assert {path.name for path in L21_INFERENCE.iterdir()} == {"queries.jsonl"}
    L21_G0 = run_prediction_variant(
        PIPELINE, L21_INFERENCE, "DEV_L21_150", "G0_E2E1_COARSE", OUTPUT_ROOT
    )
    L21_HISTORICAL_G0 = verify_historical_g0_kis(
        L21_G0, "predictions/dev_l21_150_p0_coarse_predictions.jsonl"
    )
    L21_G1 = run_prediction_variant(
        PIPELINE, L21_INFERENCE, "DEV_L21_150", "G1_COVERAGE_COARSE", OUTPUT_ROOT
    )
    L21_G2 = run_prediction_variant(
        PIPELINE, L21_INFERENCE, "DEV_L21_150", "G2_SAFE_M1", OUTPUT_ROOT
    )
    L21_RUN = combine_prediction_variants(L21_G0, L21_G1, L21_G2)
    assert all(value["validation"]["status"] == "PASS" for value in L21_RUN["variants"].values())
    PREDICTION_RUNS["DEV_L21_150"] = L21_RUN
    print(
        {
            "hashes": {variant: value["sha256"] for variant, value in L21_RUN["variants"].items()},
            "historical_g0_kis": L21_HISTORICAL_G0,
        }
    )
else:
    print("DEV_L21_150 skipped by explicit switch")

In [ ]:
if RUN_DEV_L21_150:
    # L21 GT is first loaded here, after all three L21 hashes exist.
    L21_EVAL = evaluate_finalized(L21_RUN, L21_ROOT, "DEV_L21_150", OUTPUT_ROOT)
    L21_POST = post_inference_diagnostics(L21_RUN, L21_EVAL, L21_ROOT, "DEV_L21_150", OUTPUT_ROOT)
    EVALUATIONS["DEV_L21_150"] = L21_EVAL
    POST["DEV_L21_150"] = L21_POST
    print(json.dumps({variant: value["summary"] for variant, value in L21_EVAL.items()}, indent=2))
print("TEAM_EVAL_SCORES_REPORTED_SEPARATELY=YES")

In [ ]:
from aic2026_eval.io import write_json
from triage_eg.e2eg1 import compare_variants, decisions, runtime_summary

COMPARISON = compare_variants(EVALUATIONS)
DECISION = decisions(EVALUATIONS, COMPARISON)
write_json(OUTPUT_ROOT / "diagnostics/cross_variant_comparison.json", COMPARISON["DEV_CROSS_60"])
if "DEV_L21_150" in COMPARISON:
    write_json(OUTPUT_ROOT / "diagnostics/l21_variant_comparison.json", COMPARISON["DEV_L21_150"])
RUNTIME_SUMMARY = runtime_summary(PREDICTION_RUNS, PIPELINE, STARTUP_SECONDS)
assert RUNTIME_SUMMARY["qa_opaque_machine_id_output_count"] == 0
print(json.dumps(DECISION, indent=2))

In [ ]:
from IPython.display import Image, display

from triage_eg.e2eg1 import render_cross_review

REVIEW_PATHS = render_cross_review(PIPELINE, CROSS_RUN, CROSS_EVAL, CROSS_ROOT, OUTPUT_ROOT)
assert len([path for path in REVIEW_PATHS if path.parent.name == "review"]) <= 20
for path in [value for value in REVIEW_PATHS if value.parent.name == "montages"]:
    display(Image(filename=str(path)))
print({"review_artifacts": len(REVIEW_PATHS), "GT_OVERLAY_PHASE": "POST_INFERENCE_ONLY"})

In [ ]:
from triage_eg.e2eg1 import create_bundle, write_manifests

write_manifests(
    OUTPUT_ROOT,
    pipeline=PIPELINE,
    dataset_root=DATASET_ROOT,
    team_eval_bundle=TEAM_EVAL_ZIP,
    historical_e2e1_bundle=HISTORICAL_E2E1_ZIP,
    branch=BRANCH,
    git_commit=HEAD,
    prediction_runs=PREDICTION_RUNS,
    runtime=RUNTIME_SUMMARY,
    decision=DECISION,
)
BUNDLE = create_bundle(OUTPUT_ROOT, ZIP_PATH)
with ZipFile(BUNDLE) as archive:
    BUNDLE_MEMBERS = archive.namelist()
assert not any(
    name.casefold().endswith((".mp4", ".npy", ".npz", ".pt", ".pth", ".bin"))
    for name in BUNDLE_MEMBERS
)
assert not any("sealed" in name.casefold() for name in BUNDLE_MEMBERS)
print(
    {
        "download_zip": str(BUNDLE),
        "size_bytes": BUNDLE.stat().st_size,
        "members": len(BUNDLE_MEMBERS),
    }
)

In [ ]:
from triage_eg.e2eg1 import formal_report_lines

for line in formal_report_lines(
    git_commit=HEAD,
    evaluations=EVALUATIONS,
    comparison=COMPARISON,
    post=POST,
    runtime=RUNTIME_SUMMARY,
    decision=DECISION,
    zip_path=BUNDLE,
):
    print(line)
print(
    "INPUTS_USED=",
    {
        "raw_dataset": str(DATASET_ROOT),
        "team_eval_dev_zip": str(TEAM_EVAL_ZIP),
        "team_eval_source": TEAM_EVAL_SOURCE,
        "stage1": str(STAGE1_ROOT),
        "stage1b": str(STAGE1B_ROOT),
        "stage1e": str(STAGE1E_ROOT),
        "clip": str(CLIP_ROOT),
        "opus": str(OPUS_ROOT),
        "historical_e2e1_optional": str(HISTORICAL_E2E1_ZIP) if HISTORICAL_E2E1_ZIP else None,
    },
)
PIPELINE.close()